# <span style="color:#ff8c00">02 · Train your first VQA model</span>

Visual Question Answering (VQA) requires an algorithm to answer open-ended questions about images. Given an image and an associated question, the model predicts the most appropriate answer from a predefined answer set.

This notebook trains the classical **H1** baseline: an image vector from a frozen CNN backbone and a question vector are combined via elementwise product and mapped to 1,000 candidate answers. The model is scored using the official VQA accuracy metric across the entire `val2014` split (214,354 questions).

# <span style="color:#ff8c00">Setup</span>

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import vqa_lab

ctx = vqa_lab.context()
print("Device:", ctx.device)
print(f"Fit questions:     {len(ctx.data.fit):,}")
print(f"Holdout questions: {len(ctx.data.hold):,}")
print(f"Val questions:     {len(ctx.data.val):,}")
print(f"Vocabulary:        {len(ctx.data.vocab):,} answers")

In [ ]:
IMAGE, TEXT, HEAD = "resnet50.tv_in1k", "glove", "product"  # change these

# <span style="color:#ff8c00">Train the model</span>

Train an H1 model combining `IMAGE` features and `TEXT` question embeddings with the `HEAD` fusion head.

On first use, image features are extracted and cached. The model then trains with AdamW and early stopping on the image-disjoint holdout split, taking about 5 to 10 minutes on a 24 GB GPU.

In [ ]:
row = vqa_lab.run(ctx, IMAGE, TEXT, HEAD)
df = pd.DataFrame([row])
display(df)

# <span style="color:#ff8c00">Accuracy breakdown</span>

Visualize the official VQA accuracy on the full `val2014` set, showing performance overall and across the three answer categories (`yes/no`, `number`, and `other`).

In [ ]:
metrics = ["yes_no", "number", "other", "val"]
labels = ["Yes / No", "Number", "Other", "Overall Val"]
values = [row[m] for m in metrics]

fig, ax = plt.subplots(figsize=(7, 4.5))
bars = ax.bar(labels, values, color=["#3274a1", "#e1812c", "#3a923a", "#c03d3e"], width=0.55)

ax.set_ylabel("Official VQA Accuracy (%)", fontsize=11)
ax.set_ylim(0, 100)
ax.set_title(f"{row['backbone']} + {row['text']} ({row['fusion']} head) on val2014", fontsize=12, pad=12)

for bar in bars:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, yval + 1.5, f"{yval:.2f}%", ha="center", va="bottom", fontweight="bold", fontsize=10)

ax.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

# <span style="color:#ff8c00">Clean up cached features</span>

Under the combination-first rule (every question encoder and head of one image encoder runs before the next encoder starts), an image encoder's extracted features can be deleted once its runs are done, to save disk space.

In [ ]:
vqa_lab.forget(ctx, IMAGE)
print(f"Deleted cached features for {IMAGE}.")

# <span style="color:#ff8c00">Try next</span>

Now that you have trained your first baseline model, here are directions to explore:
1. **Try another image encoder**: Pick an alternative backbone key from `configs/vqa_lab.yaml (classical.image)` (e.g. `convnext_base.clip_laion2b` or `tf_efficientnetv2_l.in21k_ft_in1k`).
2. **Use BERT question embeddings**: Pass `text="bert"` to encode questions with BERT-base hidden states (~19 GB on disk when first generated).
3. **Question-guided spatial attention**: Switch to `head="attention"` to perform attention over spatial 7×7 feature map grids.
4. **Run the full benchmark**: Open `notebooks/03_encoder_benchmark.ipynb` to run the complete sequential funnel across all candidate backbones.